<a href="https://colab.research.google.com/github/babi00/ai4biological-pattern/blob/guido-clean/invasive_plants_analysis/region_labelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Obtain the file with the prediction for all images, the clusters with the regions separated by cluster and the file with the labels for each cluster.

Generate a file containing for each region the label, the prediction and the ground truth.

In [1]:
import os
import pandas as pd
from google.colab import drive
import time
drive.mount('/content/drive/')

Mounted at /content/drive/


In [6]:
clusters = '/content/drive/MyDrive/Thesis/clustered_regions_leave_one_out_best_silhoutte_leave_one_out_kmean_30_clusters_ig'
predictions = '/content/drive/MyDrive/Thesis/full_predictions.csv'
cluster_labels = '/content/drive/MyDrive/Thesis/cluster_labels_barbara.csv'

regions_labels = '/content/drive/MyDrive/Thesis/regions_labels.csv'

predictions_df = pd.read_csv(predictions)
cluster_labels_df = pd.read_csv(cluster_labels)

In [3]:
cluster_labels_df[:5]

,cluster_id,leaf,flower,stem,hand,background/undefined
0,0,NaN,NaN,NaN,NaN,1.0
1,1,1.0,1.0,NaN,NaN,NaN
2,2,1.0,NaN,1.0,NaN,NaN
3,3,NaN,NaN,NaN,NaN,1.0
4,4,1.0,NaN,1.0,NaN,NaN


In [4]:
predictions_df[:5]

,obs_id,filename,group,y_true,y_pred
0,17946,lythrum_portula_787.jpeg,lythrum_portula,1,1
1,3738,lythrum_salicaria_3739.jpg,lythrum_salicaria,0,0
2,19547,lythrum_portula_2388.jpeg,lythrum_portula,1,1
3,34957,lythrum_alatum_248.jpg,lythrum_alatum,1,1
4,14073,lythrum_junceum_743.jpg,lythrum_junceum,1,1


In [9]:
start = time.time()

regions_df = pd.DataFrame(columns=['obs_id', 'filename', 'region', 'group', 'prediction', 'ground_truth', 'leaf', 'flower', 'stem', 'hand', 'undefined'])

for dir in os.listdir(clusters):
  cluster_id = int(dir.split('_')[1]) #id number
  labels = {
      'leaf' : cluster_labels_df[cluster_labels_df['cluster_id']==cluster_id]['leaf'].values[0],
      'flower' : cluster_labels_df[cluster_labels_df['cluster_id']==cluster_id]['flower'].values[0],
      'stem' : cluster_labels_df[cluster_labels_df['cluster_id']==cluster_id]['stem'].values[0],
      'hand' : cluster_labels_df[cluster_labels_df['cluster_id']==cluster_id]['hand'].values[0],
      'undefined' : cluster_labels_df[cluster_labels_df['cluster_id']==cluster_id]['background/undefined'].values[0]
  }
  for region in os.listdir(f'/content/drive/MyDrive/Thesis/clustered_regions_leave_one_out_best_silhoutte_leave_one_out_kmean_30_clusters_ig/{dir}'):

    filename = region.split('_')[0] + '_' + region.split('_')[1] + '_' + region.split('_')[2] #lyhtrum + _ + salicaria + _ + 4690.jpeg
    obs_id = predictions_df[predictions_df['filename']==filename]['obs_id'].values[0]
    group = predictions_df[predictions_df['filename']==filename]['group'].values[0]
    ground_truth = predictions_df[predictions_df['filename']==filename]['y_true'].values[0]
    prediction = predictions_df[predictions_df['filename']==filename]['y_pred'].values[0]

    regions_df.loc[len(regions_df)] = [obs_id, filename, region, group, prediction, ground_truth, labels['leaf'], labels['flower'], labels['stem'], labels['hand'], labels['undefined']]


end = time.time()

print(f'total time: {end-start:.3f}')

regions_df

regions_df.to_csv(regions_labels, index=False)


total time: 139.674
